## Imports, paths, y funciones auxiliares

In [32]:
# Imports iniciales
from datetime import datetime, time, timedelta
from pathlib import Path
import pandas as pd
import openpyxl

In [33]:
# Lo pongo así para que no se rompa el path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW_DIR = ROOT / "data" / "raw"
# Acepta "Totalizador_Formateado" y "Totalizadores_Formateados"
archivos = sorted(RAW_DIR.glob("Totalizador*_Formateado*.csv"))
assert archivos, f"No hay CSV en {RAW_DIR}: correr primero src/procesado_raw.py"

SEP = " | "                   # separador "Hoja | columna" que pone el procesado
CLAVES = ["DIA", "HORA", "N_REPETICION"]
HOJAS_DESCARTAR = ["Auxiliar", "Seguimiento Dia"]
ERRORES_EXCEL = ["#VALUE!", "#DIV/0!", "#N/A", "#REF!", "#NAME?", "#NUM!", "#NULL!"]
LIMITE = 23 * 3600            # 23:00:00 en segundos desde medianoche


SALIDA = ROOT / "data" / "processed" / "datos_base.csv"

In [34]:
def hoja_de(col):
    """'Consolidado EE | Frio (Kw)' -> 'Consolidado EE'. None para DIA, HORA, N_REPETICION."""
    return col.split(SEP, 1)[0] if SEP in col else None


def leer_formateado(ruta):
    # low_memory=False: con ~400 columnas de tipos mezclados pandas tipa por bloques
    df = pd.read_csv(ruta, low_memory=False, na_values=ERRORES_EXCEL,
                     dtype={"HORA": "string"})
    # El procesado escribe DIA como dd/mm/aaaa: sin format, pandas invierte día y mes
    df["DIA"] = pd.to_datetime(df["DIA"], format="%d/%m/%Y")
    return df

## Analisis inicial y Union
Primera parte del exploratorio, lo que quiero ver es como unir los 3 excels juntos y cuan mal están:

Los leo:

In [35]:
dfs = {}
for archivo in archivos:
    print("Leyendo", archivo.name)
    dfs[archivo.name] = leer_formateado(archivo)

for nombre, df in dfs.items():
    print(nombre, df.shape)

Leyendo Totalizador_Formateado 2020_2022.csv
Leyendo Totalizador_Formateado 2021_2023.csv
Leyendo Totalizador_Formateado 2022_2023.csv
Totalizador_Formateado 2020_2022.csv (15621, 344)
Totalizador_Formateado 2021_2023.csv (15342, 424)
Totalizador_Formateado 2022_2023.csv (12017, 431)


Parece que el primer excel tiene muchas menos columnas que los otros dos.

In [36]:
def a_segundos(horas):
    """Serie de horas en texto ('23:59:00') -> segundos desde las 00:00. NaN si no se entiende."""
    return pd.to_timedelta(horas, errors="coerce").dt.total_seconds()


def formato(seg):
    return f"{seg // 3600:02d}:{seg % 3600 // 60:02d}:{seg % 60:02d}"


def revisar_horas(df):
    """Horas posteriores a las 23:00:00 y valores de hora con formato raro."""
    horas = df["HORA"]
    segundos = a_segundos(horas)

    tarde = segundos[segundos > LIMITE]
    print("Horas > 23:00:00 (valor: cantidad de filas)")
    for seg, n in tarde.value_counts().sort_index().items():
        print(f"  {formato(int(seg))}: {n}")

    raros = horas[segundos.isna() & horas.notna()]
    if len(raros):
        print("Valores con formato no reconocido:")
        for valor, n in raros.astype(str).value_counts().items():
            print(f"  {valor!r}: {n}")


def revisar_dias(df, min_filas=24, max_filas=25):
    """Filas por día, y un df aparte con los días con menos de min_filas o más de max_filas."""
    conteo = df.groupby("DIA").size()

    print("Cantidad de días según filas por día:")
    print(conteo.value_counts().sort_index())

    dias_raros = conteo[(conteo < min_filas) | (conteo > max_filas)].index
    df_raros = df[df["DIA"].isin(dias_raros)].copy()
    return conteo, df_raros

Me ponga a ver como es el formato de las :59 

**queda justificar por que las :59**

In [37]:
for nombre, df in dfs.items():
    print(f"\n=== {nombre} ===")
    revisar_horas(df)


=== Totalizador_Formateado 2020_2022.csv ===
Horas > 23:00:00 (valor: cantidad de filas)
  23:00:01: 10
  23:59:00: 621
  23:59:01: 13
  23:59:59: 2
Valores con formato no reconocido:
  '01/01/1900': 24

=== Totalizador_Formateado 2021_2023.csv ===
Horas > 23:00:00 (valor: cantidad de filas)
  23:00:01: 9
  23:59:00: 606
  23:59:01: 5
  23:59:59: 1
Valores con formato no reconocido:
  '01/01/1900': 24

=== Totalizador_Formateado 2022_2023.csv ===
Horas > 23:00:00 (valor: cantidad de filas)
  23:00:01: 5
  23:59:00: 476
  23:59:01: 3
  23:59:59: 1


La mayoría de horas están bien, hay que tener en cuenta las 23:00:01 seguramente son duplicados de 23:00 y no de :59. 

In [38]:
conteos = {}
lista_raros = []

for nombre, df in dfs.items():
    print(f"\n=== {nombre} ===")
    conteos[nombre], df_raros = revisar_dias(df)
    lista_raros.append(df_raros)

raros = pd.concat(lista_raros, ignore_index=True)
print(raros.shape)


=== Totalizador_Formateado 2020_2022.csv ===
Cantidad de días según filas por día:
8        1
23       2
24       5
25     574
26      21
49       1
50       5
52       1
100      2
Name: count, dtype: int64

=== Totalizador_Formateado 2021_2023.csv ===
Cantidad de días según filas por día:
9        1
16       1
23       3
24       3
25     584
26       1
49       1
50       5
51       1
100      2
Name: count, dtype: int64

=== Totalizador_Formateado 2022_2023.csv ===
Cantidad de días según filas por día:
8        1
16       1
17       1
20       1
23       1
25     459
26       4
50       3
51       1
53       1
100      1
Name: count, dtype: int64
(2363, 450)


Hay como 100 días pegados. Saco ahora todos los valores que no sean 
- 23:59:00
- 23:59:01
- 23:59:59

In [39]:
def filtrar_2359(df):
    """Deja solo las filas con hora 23:59:xx (23:59:00, 23:59:01, 23:59:59)."""
    seg = a_segundos(df["HORA"].astype("string"))
    inicio = 23 * 3600 + 59 * 60          # 23:59:00
    fin = inicio + 59                     # 23:59:59
    return df[(seg >= inicio) & (seg <= fin)].copy()

In [40]:
dfs_2359 = {nombre: filtrar_2359(df) for nombre, df in dfs.items()}

# Veo para una tabla (solo EE, si no son ~400 columnas):
f = dfs_2359[nombre]
f.head(5)

,DIA,HORA,N_REPETICION,Consolidado KPI | EE Planta / Hl,Consolidado KPI | EE Elaboracion / Hl,Consolidado KPI | EE Bodega / Hl,Consolidado KPI | EE Cocina / Hl,Consolidado KPI | EE Envasado / Hl,Consolidado KPI | EE Linea 2 / Hl,Consolidado KPI | EE Linea 3 / Hl,...,Auxiliar | .21,Auxiliar | .22,Auxiliar | .23,Auxiliar | .24,Auxiliar | .25,Auxiliar | .26,Auxiliar | .27,Auxiliar | .28,Auxiliar | .29,Auxiliar | .30
24,2022-07-01,23:59:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
49,2022-07-02,23:59:00,0,8.040529,0.937245,0.768611,0.333626,1.720886,4.826019,2.298424,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
74,2022-07-03,23:59:00,0,6.574294,0.750594,0.577662,0.337017,1.539617,4.332178,1.922850,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
99,2022-07-04,23:59:00,0,7.766012,0.846567,0.736241,0.356453,1.890370,4.440273,1.540381,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
124,2022-07-05,23:59:00,0,6.916359,0.846227,0.683930,0.326098,1.640218,4.381815,1.549196,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
# Hojas presentes en cada archivo (según el prefijo "Hoja | ")
hojas_por_archivo = {n: {hoja_de(c) for c in df.columns} - {None} for n, df in dfs.items()}
todas = set.union(*hojas_por_archivo.values())
for n, h in hojas_por_archivo.items():
    print(f"{n}: {len(h)} hojas | faltan: {sorted(todas - h)}")

Totalizador_Formateado 2020_2022.csv: 15 hojas | faltan: ['Seguimiento Dia']
Totalizador_Formateado 2021_2023.csv: 16 hojas | faltan: []
Totalizador_Formateado 2022_2023.csv: 15 hojas | faltan: ['Seguimiento Dia']


Borro seguimiento día por que está en solo un excel y auxiliar **rellenar por que auxiliar, fabri sabía**

In [42]:
# Saco las hojas que no sirven (si no están, no pasa nada)
for n, df in dfs.items():
    cols = [c for c in df.columns if hoja_de(c) in HOJAS_DESCARTAR]
    dfs[n] = df.drop(columns=cols)
    print(f"{n}: saco {len(cols)} columnas de {HOJAS_DESCARTAR}")

Totalizador_Formateado 2020_2022.csv: saco 32 columnas de ['Auxiliar', 'Seguimiento Dia']
Totalizador_Formateado 2021_2023.csv: saco 35 columnas de ['Auxiliar', 'Seguimiento Dia']
Totalizador_Formateado 2022_2023.csv: saco 33 columnas de ['Auxiliar', 'Seguimiento Dia']


### Union de los datos

In [43]:
partes = []
for nombre, df in dfs.items():
    f = filtrar_2359(df)
    f.insert(0, "ARCHIVO", nombre)
    partes.append(f)
    print(f"{nombre:45s} {len(f):5d} filas 23:59 | {f['DIA'].nunique()} días")

# Une por nombre de columna; las que no están en algún archivo quedan NaN
df_total = pd.concat(partes, ignore_index=True)
df_total = df_total.sort_values(["DIA", "HORA", "ARCHIVO", "N_REPETICION"], kind="stable").reset_index(drop=True)

print("Unido:", df_total.shape, df_total["DIA"].min().date(), "->", df_total["DIA"].max().date())


Totalizador_Formateado 2020_2022.csv            636 filas 23:59 | 609 días
Totalizador_Formateado 2021_2023.csv            612 filas 23:59 | 599 días
Totalizador_Formateado 2022_2023.csv            480 filas 23:59 | 471 días
Unido: (1728, 413) 2020-07-01 -> 2023-10-25


In [44]:
df_total.to_csv(SALIDA, index=False, date_format="%Y-%m-%d")
print("Guardado en", SALIDA)

Guardado en c:\Users\Usuario\Cerveceria-LabDatosII\data\processed\datos_base.csv


# Analisis del DF completo

In [45]:
df_total = pd.read_csv(SALIDA, low_memory=False, parse_dates=["DIA"], dtype={"HORA": "string"})
print(df_total.shape)
df_total.head()

(1728, 413)


,ARCHIVO,DIA,HORA,N_REPETICION,Consolidado KPI | EE Planta / Hl,Consolidado KPI | EE Elaboracion / Hl,Consolidado KPI | EE Bodega / Hl,Consolidado KPI | EE Cocina / Hl,Consolidado KPI | EE Envasado / Hl,Consolidado KPI | EE Linea 2 / Hl,...,Consolidado Produccion | Unnamed: 18,Totalizadores Produccion | Hl Session IPA,Totalizadores Produccion | Id,Totalizadores Energia | Id,Totalizadores Agua | Id,Totalizadores Gas y Vapor | Id,Totalizadores Aire | Id,Totalizadores CO2 | Id,Totalizadores Efluentes | Id,"Totalizadores Glicol | Tot L3, L4 y Planta de CO2"
0,Totalizador_Formateado 2020_2022.csv,2020-07-01,23:59:00,0,642.727209,47.145349,69.023256,0.000000,13.813953,14.578784,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Totalizador_Formateado 2020_2022.csv,2020-07-02,23:59:00,0,7.767254,0.769609,0.798838,0.319229,2.358593,4.158962,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Totalizador_Formateado 2020_2022.csv,2020-07-03,23:59:00,0,8.801205,0.862593,0.835762,0.260924,1.985462,39.076667,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Totalizador_Formateado 2020_2022.csv,2020-07-04,23:59:00,0,5.175639,0.439225,0.371077,0.258048,1.442114,4.348182,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Totalizador_Formateado 2020_2022.csv,2020-07-05,23:59:00,0,7.924665,0.802365,0.717787,0.301592,1.664726,5.125920,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Para ver la cantidad de días únicos: 

In [46]:
por_dia = df_total.groupby("DIA").agg(filas=("ARCHIVO", "size"), archivos=("ARCHIVO", "nunique"))
print("Días únicos:", len(por_dia))
print("Días con más de una fila:", (por_dia["filas"] > 1).sum())
print("Días presentes en más de un archivo:", (por_dia["archivos"] > 1).sum())

Días únicos: 1185
Días con más de una fila: 516
Días presentes en más de un archivo: 494


In [47]:
df_total = df_total.drop(columns = ['HORA', 'N_REPETICION', 'ARCHIVO'])

## Analísis del tipo de dato

Para ver los tipos de datos de los diferentes valores:

In [52]:
# dtype de cada columna como texto ("float64", "int64", "object", "str", "string", "datetime64[us]"...)
tipos = df_total.dtypes.astype(str)
print(tipos.value_counts(), "\n")

print("Columnas que son 'str': \n", df_total.columns[df_total.dtypes == "str"])

float64           404
str                 5
datetime64[us]      1
Name: count, dtype: int64 

Columnas que son 'str': 
 Index(['Totalizadores Efluentes | Totalizador Bba P1',
       'Totalizadores Efluentes | Totalizador Bba P2',
       'Totalizadores Efluentes | Totalizador Bba P4',
       'Totalizadores Efluentes | Totalizador Bba Envasado',
       'Totalizadores Efluentes | Totalizador Bba P51'],
      dtype='str')


Hay
- Datetime - es el día.

Quedan encontrar las columnas que tienen como datso:
- 5 str

In [ ]:
# Me quedo con las que no son numéricas ni fecha
es_numerica = tipos.isin(["float64"])
es_fecha = tipos.str.startswith("datetime")
cols_texto = df_total.select_dtypes(exclude=["number", "datetime"]).columns

print(f"{len(cols_texto)} columnas no numéricas:")
print(tipos[cols_texto].sort_values().to_string())

5 columnas no numéricas:
Totalizadores Efluentes | Totalizador Bba P1          str
Totalizadores Efluentes | Totalizador Bba P2          str
Totalizadores Efluentes | Totalizador Bba P4          str
Totalizadores Efluentes | Totalizador Bba Envasado    str
Totalizadores Efluentes | Totalizador Bba P51         str


- La columna ARCHIVO la puedo sacar, pensé que iba a ser de más ayuda para al analisis (la dejo aparte en caso que al terminemos usanddo). 
- Hay varios totalizadores de efluentes que son objetos, ver por que y seguramente pasar a float.

In [ ]:
nulos = pd.DataFrame({
    "dtype": df_total[cols_texto].dtypes.astype(str),
    "nulos": df_total[cols_texto].isna().sum(),
    "no_nulos": df_total[cols_texto].notna().sum(),
})
nulos["pct_nulos"] = (nulos["xnulos"] / len(df_total) * 100).round(2)

nulos.sort_values("pct_nulos", ascending=False)


,dtype,nulos,no_nulos,pct_nulos
Totalizadores Efluentes | Totalizador Bba P1,str,16,1712,0.93
Totalizadores Efluentes | Totalizador Bba P2,str,16,1712,0.93
Totalizadores Efluentes | Totalizador Bba P4,str,16,1712,0.93
Totalizadores Efluentes | Totalizador Bba Envasado,str,16,1712,0.93
Totalizadores Efluentes | Totalizador Bba P51,str,16,1712,0.93


In [ ]:
for c in cols_texto:
    antes = df_total[c].notna().sum()
    convertida = pd.to_numeric(df_total[c], errors="coerce").astype("float64")
    perdidos = antes - convertida.notna().sum()
    if perdidos:
        no_conv = df_total.loc[df_total[c].notna() & convertida.isna(), c]
        print(f"{c}: {perdidos} valores pasan a NaN -> {no_conv.astype(str).value_counts().head(5).to_dict()}")
    df_total[c] = convertida

print(df_total[cols_texto].dtypes)

Totalizadores Efluentes | Totalizador Bba P1: 1 valores pasan a NaN -> {'407,3': 1}
Totalizadores Efluentes | Totalizador Bba P2: 1 valores pasan a NaN -> {'1407,2': 1}
Totalizadores Efluentes | Totalizador Bba P4: 1 valores pasan a NaN -> {'617,8': 1}
Totalizadores Efluentes | Totalizador Bba Envasado: 1 valores pasan a NaN -> {'497,6': 1}
Totalizadores Efluentes | Totalizador Bba P51: 1 valores pasan a NaN -> {'617,8': 1}
Totalizadores Efluentes | Totalizador Bba P1          float64
Totalizadores Efluentes | Totalizador Bba P2          float64
Totalizadores Efluentes | Totalizador Bba P4          float64
Totalizadores Efluentes | Totalizador Bba Envasado    float64
Totalizadores Efluentes | Totalizador Bba P51         float64
dtype: object


## Tratamiento de los unnameds 

Al ver el dataset, encontramos muchas columnas llamadas unnamed (numero). Puede que haya alguna que tenga valores de verdad, y otras que simplemente son errores. Para descartar los errores, vamos a hacer un df aparte con los unnameds y analizar el porcentaje de nulos:

In [ ]:
# Columnas unnamed (vienen como "Hoja | Unnamed: N")
cols_unnamed = [c for c in df_total.columns if "Unnamed" in c]

unnamed = pd.DataFrame({
    "hoja": [hoja_de(c) for c in cols_unnamed],
    "nulos": df_total[cols_unnamed].isna().sum(),
}, index=cols_unnamed)

unnamed["pct_nulos"] = (unnamed["nulos"] / len(df_total) * 100).round(2)
unnamed = unnamed.sort_values("pct_nulos")

print(f"{len(unnamed)} columnas unnamed")
unnamed

24 columnas unnamed


,hoja,nulos,pct_nulos
Consolidado Produccion | Unnamed: 15,Consolidado Produccion,1685,97.51
Consolidado Produccion | Unnamed: 16,Consolidado Produccion,1685,97.51
Consolidado Produccion | Unnamed: 17,Consolidado Produccion,1685,97.51
Consolidado Produccion | Unnamed: 18,Consolidado Produccion,1685,97.51
Totalizadores Energia | Unnamed: 53,Totalizadores Energia,1697,98.21
Totalizadores Energia | Unnamed: 54,Totalizadores Energia,1727,99.94
Totalizadores Energia | Unnamed: 56,Totalizadores Energia,1727,99.94
Totalizadores Energia | Unnamed: 55,Totalizadores Energia,1727,99.94
Totalizadores Energia | Unnamed: 57,Totalizadores Energia,1727,99.94
Totalizadores Energia | Unnamed: 58,Totalizadores Energia,1727,99.94


In [ ]:
df_recortado = df_total.drop(columns=unnamed.index)
print(f"Saqué {len(unnamed)} columnas unnamed -> {df_recortado.shape}")

Saqué 24 columnas unnamed -> (1728, 386)


## Tratado de duplicados y regla de desempate

Vamos a tomar como criterio de desempate: 
- Nos quedamos con la fila con mayor cantidad de valores nulos, y en caso que la fila descartada tenga algún valor real que para la fila guardada es nula, se va a rellenar. 

In [59]:
# Días que aparecen en más de una fila (se cuenta la clave DIA, no las filas)
df_dup = df_recortado.copy()
dias_repetidos = df_dup.loc[df_dup["DIA"].duplicated(keep=False), "DIA"].drop_duplicates()

print("Total de días repetidos:", len(dias_repetidos), "\n")
#print(df_dup[df_dup["DIA"].duplicated() == True])


Total de días repetidos: 516 



In [62]:
cols_datos = df_dup.columns.drop("DIA")

# Cantidad de nulos de cada fila; _orden guarda el orden original para el desempate
df_dup["_nulos"] = df_dup[cols_datos].isna().sum(axis=1)        
df_dup["_orden"] = range(len(df_dup))   

# Dentro de cada día: primero la fila con menos nulos; si empatan, la que venía primero
ordenado = df_dup.sort_values(["DIA", "_nulos", "_orden"])

print(ordenado[["DIA", "_nulos", "_orden"]].head(10))
# first() toma, por columna, el primer valor no nulo del grupo:
# la fila base conserva sus valores y sus huecos se rellenan con las siguientes
df_unido = ordenado.groupby("DIA", sort=True)[list(cols_datos)].first().reset_index()

assert df_unido["DIA"].is_unique
assert len(df_unido) == df_dup["DIA"].nunique()

         DIA  _nulos  _orden
0 2020-07-01      91       0
1 2020-07-02      91       1
2 2020-07-03      91       2
3 2020-07-04      91       3
4 2020-07-05      91       4
5 2020-07-06      91       5
6 2020-07-07      91       6
7 2020-07-08      91       7
8 2020-07-09      91       8
9 2020-07-10      91       9


Guardo el .csv (sobrescribo el anterior).

In [ ]:
df_unido[df_recortado.columns].to_csv(SALIDA, index=False, date_format="%Y-%m-%d")
print("Guardado en", SALIDA)

Guardado en c:\Users\Usuario\Cerveceria-LabDatosII\data\processed\datos_base.csv


In [ ]:
df_unido.info()

<class 'pandas.DataFrame'>
RangeIndex: 1185 entries, 0 to 1184
Columns: 386 entries, DIA to Totalizadores Glicol | Tot L3, L4 y Planta de CO2
dtypes: datetime64[us](1), float64(385)
memory usage: 3.5 MB


El info indica que tiene 1185 entradas. Corriendo los calculos del 2020-2023, tendrían que ser 1095 días. Hay gato encerrado. 

Revisando en un excel temporal, vemos que el menor dato es del 12/08/2020 y el mayor dato es del 30/09/2023. Hay 1144 días entremedio.
Vamos a empezar a hacer gráficos y ver que está pasando.

---
---
---
---
---
---
---
---
---
---
---
---

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict

# Si df_total tiene columnas MultiIndex (hoja, columna), las aplano a solo el nombre
if isinstance(df_total.columns, pd.MultiIndex):
    df_total.columns = df_total.columns.get_level_values(-1)

# Solo numéricas y con variación (las constantes dan falsos positivos)
X = df_total.select_dtypes("number")
X = X.loc[:, (X.nunique() > 1).values]
X = X.loc[:, ~X.columns.astype(str).str.startswith("Unnamed")]

# Etiqueta única por columna: "nombre [posición]"
X.columns = [f"{c} [{i}]" for i, c in enumerate(X.columns)]
print(X.shape)

(1728, 367)


In [ ]:
base = pd.Series(X.columns).str.replace(r"\s*\[\d+\]$", "", regex=True).str.lower().str.strip()
rep = base[base.duplicated(keep=False)]
print(rep.groupby(rep).apply(lambda s: list(X.columns[s.index])).to_string())

Series([], )


In [ ]:
grupos = defaultdict(list)
for col in X.columns:
    h = pd.util.hash_pandas_object(X[col].round(4), index=False).sum()
    grupos[h].append(col)

identicas = [cols for cols in grupos.values() if len(cols) > 1]
for cols in identicas:
    print(cols)
print(f"\n{len(identicas)} grupos de columnas idénticas")

['Consolidado Produccion | Hl Cerveza L2 [59]', 'Totalizadores Produccion | Hl Cerveza L2 [84]']
['Consolidado Produccion | Hl Cerveza L3 [60]', 'Totalizadores Produccion | Hl Cerveza L3 [85]']
['Consolidado Produccion | Hl Cerveza L4 [61]', 'Totalizadores Produccion | Hl Cerveza L4 [86]']
['Consolidado Produccion | Hl Cerveza L5 [62]', 'Totalizadores Produccion | Hl Cerveza L5 [87]']
['Consolidado Produccion | Cocimientos Diarios [63]', 'Totalizadores Produccion | Cocimientos Diarios [88]']
['Consolidado EE | KW Gral Planta [110]', 'Totalizadores Energia | KW Gral Planta [111]']
['Consolidado Agua | Red L1 y L2 [169]', 'Totalizadores Agua | Red L1 y L2 [176]']
['Consolidado Agua | FC L1 y L2 [170]', 'Totalizadores Agua | FC L1 y L2 [177]']
['Consolidado Agua | FC Lavadora L2 [171]', 'Totalizadores Agua | FC Lavadora L2 [202]']
['Consolidado Agua | Red L3 [172]', 'Totalizadores Agua | Red L3 [179]']
['Consolidado Agua | FC Lavadora L3 [173]', 'Totalizadores Agua | FC Lavadora L3 [200]'

In [ ]:
corr = X.corr()
mask = np.triu(np.ones(corr.shape, dtype=bool), k=1)
pares = corr.where(mask).stack().reset_index()
pares.columns = ["a", "b", "r"]

umbral = 0.999                         # bajalo a 0.98 para ver las "demasiado parecidas"
pares = pares[pares["r"].abs() > umbral].sort_values("r", ascending=False)
print(len(pares), "pares con |r| >", umbral)
print(pares.to_string(index=False))

584 pares con |r| > 0.999
                                                   a                                                         b         r
                       Totalizadores Aire | Id [363]                        Totalizadores Efluentes | Id [365]  1.000000
                Totalizadores Gas y Vapor | Id [362]                        Totalizadores Efluentes | Id [365]  1.000000
                Totalizadores Gas y Vapor | Id [362]                             Totalizadores Aire | Id [363]  1.000000
          Consolidado GasVapor | Vapor_L5 (KG) [346]           Totalizadores Gas y Vapor | Vapor_L5 (KG) [350]  1.000000
                Consolidado KPI | Meta Aire L4 [335]               Consolidado KPI | Meta Aire Servicios [337]  1.000000
                Consolidado KPI | Meta Aire L3 [334]               Consolidado KPI | Meta Aire Servicios [337]  1.000000
                Consolidado KPI | Meta Aire L3 [334]                      Consolidado KPI | Meta Aire L5 [336]  1.000000
      